# 14.5 从变分推导看 DDPM：ELBO 从哪里来

jshn9515  
2026-04-02

<a href="https://colab.research.google.com/github/jshn9515/dnnl-notebooks/blob/main/zh/ch14-diffusion-models/ch14.5-elbo-for-ddpm.ipynb" data-fig-align="left"><img src="https://colab.research.google.com/assets/colab-badge.svg" /></a>

前面几节里，我们已经从直觉和算法实现的角度理解了 DDPM。但是，很多读者会有一个新的疑问：

> **它们到底是经验上好用的小技巧，还是能从概率建模里严格推出来？**

这一节我们就来回答这个问题。我们会看到，DDPM 背后有一套非常标准的概率建模视角：

- 我们定义了一个带潜变量的生成模型；
- 由于直接最大化数据似然很难，于是引入 **ELBO (Evidence Lower Bound)**；
- 然后再利用前向加噪过程的特殊结构，把这个 ELBO 一步一步化简；
- 最后得到的训练目标，和我们前面看到的预测噪声的 MSE 是紧密相关的。

所以，DDPM 不是凭空发明了一个去噪损失，而是可以从变分推断的角度自然地推出来。

## 14.5.1 为什么要引入 ELBO？

我们先来回忆一下生成模型最根本的目标。无论是 AE、VAE 还是 Diffusion，我们最终都希望模型能够逼近真实数据分布 $p_{\text{data}}(x)$。更具体一点，我们希望最大化观测数据的对数似然：

$$\log p_\theta(x_0)$$

这里的 $x_0$ 表示真实样本，$\theta$ 表示模型参数[1]。

问题在于，在 DDPM 里，我们并不是直接定义一个简单的 $p_\theta(x_0)$。相反，我们引入了一整串潜变量：

$$x_1, x_2, \dots, x_T$$

于是，整个生成过程可以写成：

$$p_\theta(x_{0:T}) = p(x_T)\prod_{t=1}^T p_\theta(x_{t-1}\mid x_t)$$

其中，$p(x_T)$ 通常设成标准高斯分布；$p_\theta(x_{t-1}\mid x_t)$ 是模型需要学习的反向去噪分布。

这样定义以后，边缘似然就变成：

$$p_\theta(x_0) = \int p_\theta(x_{0:T})\,dx_{1:T}$$

你会发现，问题一下子变难了：因为我们要把所有中间变量都积分掉，这通常是很难直接算的。所以这时，一个非常熟悉的思路就出现了：

> **既然直接最大化 $\log p_\theta(x_0)$ 很难，那就去优化它的一个下界。**

是不是和 VAE 的思路很像？而这个下界，就是 ELBO。

## 14.5.2 DDPM 里的变分分布是谁？

在 VAE 里，我们引入了一个近似后验 $q_\phi(z\mid x)$，这个近似后验由编码器来扮演。编码器把图像编码成潜变量 $z$，这个过程就是我们构造的变分分布 $q$。而在 DDPM 里，对应的角色则由前向加噪过程来扮演：

$$q(x_{1:T}\mid x_0)=\prod_{t=1}^T q(x_t\mid x_{t-1})$$

这里的每一步前向转移都是我们自己规定好的，比如：

$$q(x_t\mid x_{t-1})=\mathcal{N}\Bigl(x_t;\sqrt{1-\beta_t}\,x_{t-1},\,\beta_t I\Bigr)$$

这个过程有两个重要特点：

1.  它完全已知，不需要学习；
2.  它很容易采样，也有很好的解析性质。

所以在 DDPM 里，前向过程一方面是把数据变成噪声的机制，另一方面也恰好充当了变分推断里的辅助分布 $q$。那么，它是怎么做到的呢？

## 14.5.3 从对数似然到 ELBO

现在我们来写标准的变分推导。

我们关心的是：

$$\log p_\theta(x_0)$$

对任意一个分布 $q(x_{1:T}\mid x_0)$，都有：

$$\log p_\theta(x_0) =
\log \int q(x_{1:T}\mid x_0)\frac{p_\theta(x_{0:T})}{q(x_{1:T}\mid x_0)}\,dx_{1:T}$$

因为 $\log$ 是一个凹函数，根据 Jensen 不等式：

$$\log \mathbb{E}[Z] \ge \mathbb{E}[\log Z]$$

我们有：

$$\log p_\theta(x_0) \ge \mathbb{E}_{q(x_{1:T}\mid x_0)}
\left[\log \frac{p_\theta(x_{0:T})}{q(x_{1:T}\mid x_0)} \right]$$

这就是 ELBO：

$$\mathcal{L}_{\text{ELBO}} = \mathbb{E}_{q(x_{1:T}\mid x_0)}
\left[\log p_\theta(x_{0:T})-\log q(x_{1:T}\mid x_0)\right]$$

所以 DDPM 的训练，也可以理解成最大化数据对数似然的一个可计算下界。

到这里，其实一切都还只是标准套路。真正有意思的地方在于，由于 DDPM 的前向过程和反向过程都有特殊结构，这个 ELBO 还能继续被拆开，变成一组更具体的项。

现在我们把联合分布写开：

$$p_\theta(x_{0:T}) = p(x_T)\prod_{t=1}^T p_\theta(x_{t-1}\mid x_t)$$

同时，前向过程是：

$$q(x_{1:T}\mid x_0)=\prod_{t=1}^T q(x_t\mid x_{t-1})$$

把它们代回 ELBO：

$$\mathcal{L}_{\text{ELBO}} = \mathbb{E}_q
\left[\log p(x_T)+\sum_{t=1}^T \log p_\theta(x_{t-1}\mid x_t) -
\sum_{t=1}^T \log q(x_t\mid x_{t-1})\right]$$

接下来通过整理项，可以把它改写成标准形式。在 DDPM 文献里，常见的写法是把负 ELBO 分解成三类项：

$$L = L_0 + \sum_{t=2}^T L_{t-1} + L_T$$

我们大致可以这样理解：

- $L_0$：重建项，模型能不能把图像还原出来；
- $L_{t-1}$：中间 KL 项，模型学的反向分布和真实后验之间的差距；
- $L_T$：先验匹配项，最终的噪声分布是不是接近标准高斯分布。

也就是说，DDPM 的学习目标其实不是一个黑箱损失，而是由一串很自然的概率约束组成的：

1.  最后要能把噪声还原成图像；
2.  中间每一步的反向分布要接近真实后验；
3.  最后一步的噪声分布要和标准高斯对齐。

这里最关键的是中间项。它的形式是：

$$D_\mathrm{KL}\Bigl(q(x_{t-1}\mid x_t, x_0) \;\|\; p_\theta(x_{t-1}\mid x_t)\Bigr)$$

也就是说，给定当前含噪图像 $x_t$，以及真实的干净图像 $x_0$，前向过程会产生一个真实后验 $q(x_{t-1}\mid x_t, x_0)$；我们希望模型学到的反向分布 $p_\theta(x_{t-1}\mid x_t)$ 尽可能接近它。对于每一个时间步，我们希望模型去逼近“如果我知道真实图像，那么这一步应该怎么往回走”的最优条件分布。这就把反向去噪这件事，变成了一个非常标准的分布拟合问题。

那么，MSE 是怎么从 KL 项里出来的呢？

## 14.5.4 KL 项为什么能变成 MSE？

从 14.3.1 我们知道，真实后验是一个高斯分布：

$$q(x_{t-1}\mid x_t, x_0) =
\mathcal{N}\bigl(x_{t-1};\tilde{\mu}_t(x_t,x_0),\,\tilde{\beta}_t I\bigr)$$

那么一个自然的做法就是让模型也输出一个高斯分布：

$$p_\theta(x_{t-1}\mid x_t) 
\mathcal{N}\bigl(x_{t-1};\mu_\theta(x_t,t),\,\Sigma_\theta(x_t,t)\bigr)$$

这样，中间的 KL 项就变成了两个高斯分布之间的距离。

在最经典的 DDPM 设定里，通常会把方差部分固定或部分固定：

$$\Sigma_\theta(x_t,t) = \sigma_t^2 I$$

于是模型的主要任务就集中在学习均值 $\mu_\theta(x_t,t)$ 上。

实际上，如果两个高斯协方差相同，KL 会化成一个均值的二次项加上一个常数：

$$D_\mathrm{KL}\Bigl(\mathcal{N}(\tilde{\mu}_t, \sigma_t^2 I) \;\|\;
\mathcal{N}(\mu_\theta, \sigma^2_t I)\Bigr) \propto \frac{1}{2\sigma_t^2}\|
\tilde{\mu}_t - \mu_\theta\|^2$$

所以最小化中间的 KL 项就等价于让模型最小化：

$$\|\tilde{\mu}_t(x_t,x_0) - \mu_\theta(x_t,t)\|^2$$

也就是说，从 ELBO 出发，我们先先得到的是均值匹配的平方误差。

但是我们知道，直接让模型预测均值不如让模型预测噪声来得稳定。所以我们不直接输出 $\mu_\theta(x_t,t)$，而是让模型预测噪声 $\epsilon_\theta(x_t,t)$，再用它来构造模型均值：

$$\mu_\theta(x_t,t) = \frac{1}{\sqrt{\alpha_t}}
\Bigl(x_t-\frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}} \epsilon_\theta(x_t,t)\Bigr)$$

我们把真实均值也写出来：

$$\tilde{\mu}_t(x_t,x_0) = \frac{1}{\sqrt{\alpha_t}}
\Bigl(x_t-\frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}} \epsilon\Bigr)$$

你会发现，模型估计的均值和真实均值的形式一模一样，只不过把真实噪声 $\epsilon$ 换成了模型预测的噪声 $\epsilon_\theta(x_t,t)$。因为两边的公式形式相同，所以 $\tilde{\mu}_t - \mu_\theta$ 最后只差在 $\epsilon - \epsilon_\theta(x_t,t)$ 这一项上。这样，中间的 KL 项就会化成：

$$L_{t-1} = \mathbb{E}_{x_0,\epsilon,t}
\left[\frac{\beta_t^2}{2\sigma_t^2\alpha_t(1-\bar{\alpha}_t)} \|
\epsilon - \epsilon_\theta(x_t,t)\|^2\right] + C$$

其中，$C$ 是一个和模型参数 $\theta$ 无关的常数。

论文和实际实现里，通常把上面的权重先忽略掉，或者做成简单采样平均，于是得到：

$$L_{\text{simple}} = \mathbb{E}_{x_0,\epsilon,t}
\left[\|\epsilon - \epsilon_\theta(x_t,t)\|^2\right]$$

其中，$x_t$ 是通过前向过程采样得到的：

$$x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1-\bar{\alpha}_t} \epsilon$$

这就是最经典的 DDPM 训练目标。

到此，我们从 ELBO 出发，严格地推导出了 DDPM 的训练目标。最后我们可以看到，我们最大化 ELBO 的过程，等价于最小化模型预测的噪声和真实噪声之间的 MSE。这就是 DDPM 里 ELBO 的来源。

## 14.5.5 为什么训练时只优化中间项？

相信到这里，你肯定会有个疑问：我们只优化了中间的 KL 项啊，那还有前面那个重建项和最后那个先验匹配项呢？它们不重要吗？

它们当然重要。只是我们在实际训练中，发现优化中间的 KL 项已经足够了。前面那个重建项 $L_0$ 和最后那个先验匹配项 $L_T$，虽然理论上也应该优化，但它们对最终性能的提升并不明显，所以很多实现里就直接忽略了。

比如先验匹配项：

$$L_T = D_\mathrm{KL}\Bigl(q(x_T\mid x_0) \;\|\; p(x_T)\Bigr)$$

其实，在前向加噪的过程中，可以证明的是，只要加噪的链条足够长，$q(x_T\mid x_0)$ 就会非常接近标准高斯分布 $p(x_T)$。也就是说，我们只要保证步数足够长就行，根本用不着模型去学习这个先验匹配项。

那么重建项呢？它的形式是：

$$L_0 = -\mathbb{E}_{q(x_{1:T}\mid x_0)} \left[\log p_\theta(x_0\mid x_1)\right]$$

也就是采样过程中从 $x_1$ 还原 $x_0$ 的重建误差。虽然它很重要，但真正决定模型能否学好的，是中间的 KL 项。因为它直接约束了模型在每一步的去噪能力，而重建项只是约束了最后一步的还原能力。只要模型在每一步都能学好去噪，最后一步的还原自然也就做好了。所以我们在实际训练中，通常会把重建项放在一边，专注于优化中间的 KL 项。

当然了，我们的这个 ELBO 推导省略了很多中间的过程。完整的 ELBO 涉及到求解一个非常复杂的积分，里面有很多细节需要处理。我们在这里主要是想给大家一个大致的思路，告诉大家 DDPM 的训练目标是怎么从概率建模的角度推导出来的。对于完整的推导过程，大家可以参考 (Luo 2022) 这篇综述文章，里面有非常详细的 ELBO 推导和相关的数学细节。

## 14.5.6 本章小结

这一节我们回答了一个关键问题：DDPM 的训练目标，为什么能和 ELBO 联系起来？

核心逻辑是这样的：

1.  我们想最大化数据似然 $\log p_\theta(x_0)$；
2.  由于中间有整条潜变量链，直接求解很难；
3.  于是引入前向加噪分布 $q(x_{1:T}\mid x_0)$，构造 ELBO；
4.  ELBO 展开后，会出现一系列真实后验与模型反向分布之间的 KL 项；
5.  利用前向过程的线性高斯结构，这些项可以被解析处理；
6.  最后再把模型参数化成噪声预测，就得到常见的 MSE 训练目标。

到此，DDPM 也就告一段落了。但是，我们结束了吗？当然没有。

还记得 DDPM 的采样过程吗？我们在每一步中都加入了一点噪声。这个噪声保证了生成图像的多样性，但它也带来了一些问题。比如，采样速度慢，生成质量不稳定，等等。因此，有研究者提出了一种不需要加入噪声就能进行采样的方法，叫做 **DDIM (Denoising Diffusion Implicit Models)** (Song et al. 2022)。我们在下一节里就来看看 DDIM 是怎么做到的，以及它和 DDPM 之间的关系。

Luo, Calvin. 2022. *Understanding Diffusion Models: A Unified Perspective*. <https://arxiv.org/abs/2208.11970>.

Song, Jiaming, Chenlin Meng, and Stefano Ermon. 2022. *Denoising Diffusion Implicit Models*. <https://arxiv.org/abs/2010.02502>.

[1] 关于为什么可以写成最大化对数似然，而不是最小化 KL 散度，可以参考 13.2.2 节里的推导。